### Kimi's KDA with Dense Network  


In [2]:

import os
from dataclasses import dataclass, field, asdict
from typing import Optional
import torch


# -----------------------------------------------------------------------------
# Configuration & Paths
# -----------------------------------------------------------------------------

@dataclass
class ProjectPaths:
    """Centralized configuration for absolute file paths."""
    base_dir: str = os.getcwd() #"./" # Change this to your root production directory

    @property
    def tokenizer_dir(self) -> str:
        return os.path.join(self.base_dir, "custom_tokenizer_1")

    @property
    def model_checkpoints_dir(self) -> str:
        return os.path.join(self.base_dir, "trained_models/Kimi-KDA_v1")

    @property
    def data_corpus(self) -> str:
        return os.path.join(self.base_dir, "dataset/short", "corpus_B.txt")


## @dataclass
# class MoEArgs:
#     """Configuration specific to the Mixture of Experts layers."""
#     num_experts: int = 2
#     top_k_experts: int = 1
#     z_loss_coef: float = 0.001
#     moe_loss_coef: float = 0.01
#     hidden_dim_multiplier_perexp: float = 1.0
#     first_moe_layer: int = 0
#     num_shared_experts: int = 1
#     use_shared_expert: bool = False


@dataclass
class ModelArgs:
    """Unified configuration for the Transformer architecture."""
    vocab_size: int = 10000
    d_model: int = 1024 #7168
    num_layers: int = 5 #93
    num_heads: int = 8 #96
    context_length: int = 1024
    head_dim: int = 64
    ff_hidden_dim:int = 4096

    # RoPE / NoPE
    apply_rope: bool = False     # Kimi K3 uses NoPE for KDA layers
    use_checkpointing: bool = False

    device: torch.device = field(
        default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

# -----------------------------------------------------------------------------
# Training Configuration
# -----------------------------------------------------------------------------

@dataclass
class TrainingArgs:
    """Strictly training-loop specific parameters."""
    tot_steps:int = 0  # Make it Zero for full run
    epochs: int = 2
    batch_size: int = 2
    grad_accum_steps: int = 1
    lr_rate: float = 0.0005
    grad_clip: float = 1.0
    warmup_steps: int = 100
    save_every_steps: int = 500


In [3]:
## utils

# -----------------------------------------------------------------------------
# Data Processing & Utilities
# -----------------------------------------------------------------------------

from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset
from transformers import AutoTokenizer


## Text Dataset & Memory-Efficient Streaming Utility
class TextTokenDataset(Dataset):
    """Memory-friendly dataset that reads chunks of text from a corpus."""
    def __init__(self, file_path: str, context_length: int, tokenizer_dir: str):
        self.context_length = context_length

        # Real-world fallback logic if no production custom tokenizer exists yet
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
        except Exception:
            print("Custom Tokenizer path missing or invalid. Falling back to gpt2 char/byte simulation...")
            from transformers import GPT2TokenizerFast
            self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

        if not os.path.exists(file_path):
            raise FileExistsError("Source DATA file not found !")

        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        self.tokens = self.tokenizer.encode(raw_text)
        self.num_samples = max(0, len(self.tokens) - context_length - 1)

        tokens = torch.tensor(self.tokens, dtype=torch.long)
        self.safe_vocab_size = max(len(self.tokenizer), tokens.max().item() + 1)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Grab token sequences offset by 1 for casual autoregressive language modeling targets
        chunk = self.tokens[idx : idx + self.context_length + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

## Learning Rate Schedule Factory (Cosine with Warmup)
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(1e-5, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)


def calculate_active_params_MoE(model):
    """Calculates total, active and shared parameters for a model using hybrid sparse MoE layers
    """
    total_params = 0
    routed_exp_param = 0
    shared_exp_param = 0

    num_experts = None
    num_routed_experts = None
    top_k = None
    shared_expert = False

    # 1. Safely calculate total parameters
    for param in model.parameters():
        if param.requires_grad:
            total_params += param.numel()

    # 2. Iterate strictly through modules to prevent string-matching collisions with dense layers
    for name, module in model.named_modules():
        # --- Standard MoE Check ---
        if isinstance(module, MoEFeedForward):
            if num_experts is None:
                num_experts = module.num_experts
                top_k = module.num_experts_per_tok

            # These are batched nn.Parameters, so we check them directly
            for p_name, param in module.named_parameters(recurse=False):
                if p_name in ["w1", "w2", "w3"] and param.requires_grad:
                    routed_exp_param += param.numel()
        # --- DeepSeek Sparse MoE Check ---
        elif isinstance(module, DeepSeekSparseMoE):
            if num_routed_experts is None:
                num_routed_experts = module.n_routed_experts
                top_k = module.num_experts_per_tok
                shared_expert = True

            # 1. Count Routed Experts (batched nn.Parameters)
            for p_name, param in module.named_parameters(recurse=False):
                if p_name in ["w1", "w2", "w3"] and param.requires_grad:
                    routed_exp_param += param.numel()

            # 2. Count Shared Experts (nn.Linear layers)
            for shared_module in [module.shared_w1, module.shared_w2, module.shared_w3]:
                for param in shared_module.parameters():
                    if param.requires_grad:
                        shared_exp_param += param.numel()

    # Safety check if no MoE layer was found
    if (num_experts is None) and (num_routed_experts is None):
        raise ValueError("Could not find any MoEFeedForward or DeepSeekSparseMoE layers in the model.")

    total_expert_params = routed_exp_param + shared_exp_param
    base_params = total_params - total_expert_params

    # 3. Math to find active parameters
    if shared_expert:
        # Divide routed expert parameters by the number of experts to find 1 routed expert's size
        # (This scales perfectly even if you have multiple MoE layers)
        single_routedexp_size = routed_exp_param // num_routed_experts

        # DeepSeek Rule: ALL shared experts are active for EVERY token.
        active_shared = shared_exp_param
        active_routed = single_routedexp_size * top_k

        active_params = base_params + active_shared + active_routed

        return {
            "total_parameters": total_params/1e6, # in Million
            "active_parameters": active_params/1e6,
            "base_parameters": base_params/1e6, # Non-MoE params (Embeddings, Attn, Dense Layers)
            "shared_expert_parameters": shared_exp_param/1e6,
            "routed_expert_parameters": routed_exp_param/1e6,
        }
    else:
        # Standard MoE
        single_expert_size = total_expert_params // num_experts
        active_params = base_params + (single_expert_size * top_k)

        return {
            "total_parameters": total_params/1e6,
            "active_parameters": active_params/1e6,
            "base_parameters": base_params/1e6,
            "routed_expert_parameters": total_expert_params/1e6,
        }



In [ ]:
## Model
## Added weight absorption and MoE integration

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple


class ShortConv1d(nn.Module):
    """Causal 1D Depthwise Convolution with state caching for O(1) inference."""
    def __init__(self, dim: int, kernel_size: int = 4):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(
            in_channels=dim,
            out_channels=dim,
            kernel_size=kernel_size,
            groups=dim,  # Depthwise
            bias=False
        )

    def forward(self, x: torch.Tensor, conv_state: torch.Tensor = None):
        B, T, D = x.shape
        x_trans = x.transpose(1, 2)  # (B, D, T)

        if conv_state is None:
            # Prefill or Training: Pad with zeros
            x_padded = F.pad(x_trans, (self.kernel_size - 1, 0))
        else:
            # Decoding: Prepend the cached state from previous tokens
            x_padded = torch.cat([conv_state, x_trans], dim=-1)

        out = self.conv(x_padded)

        # Cache the last (kernel_size - 1) tokens for the next inference step
        next_conv_state = x_padded[..., -(self.kernel_size - 1):].detach()

        return out.transpose(1, 2), next_conv_state

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

def l2_norm(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """L2 Normalization along the head vector dimension."""
    return F.normalize(x, p=2, dim=-1, eps=eps)

def kda_recurrent(
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        g: torch.Tensor,
        beta: torch.Tensor,
        initial_state: Optional[torch.Tensor] = None,
        q_scale: float = 1.0,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
    B, T, H, K = q.shape
    V = v.shape[-1]
    orig_dtype = v.dtype

    q = q.float() * q_scale
    k = k.float()
    v = v.float()
    g = g.float()
    beta = beta.float().squeeze(-1)

    if initial_state is None:
        state = torch.zeros(B, H, K, V, device=q.device, dtype=torch.float32)
    else:
        state = initial_state.float()

    outputs = []

    for t in range(T):
        q_t = q[:, t]
        k_t = k[:, t]
        v_t = v[:, t]
        g_t = g[:, t]
        beta_t = beta[:, t]

        state = state * torch.exp(g_t).unsqueeze(-1)
        v_hat = torch.einsum("bhk,bhkv->bhv", k_t, state)
        error = v_t - v_hat

        state = state + torch.einsum("bh,bhk,bhv->bhkv", beta_t, k_t, error)
        out_t = torch.einsum("bhk,bhkv->bhv", q_t, state)
        outputs.append(out_t)

    out = torch.stack(outputs, dim=1)
    return out.to(orig_dtype), state.to(orig_dtype)

def kda_chunkwise(
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        g: torch.Tensor,
        beta: torch.Tensor,
        initial_state: Optional[torch.Tensor] = None,
        chunk_size: int = 64,
        q_scale: float = 1.0,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
    B, T, H, K = q.shape
    V = v.shape[-1]
    orig_dtype = v.dtype

    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")

    q = q.float() * q_scale
    k = k.float()
    v = v.float()
    g = g.float()
    beta = beta.float()
    if beta.ndim == 4:
        beta = beta.squeeze(-1)

    num_chunks = (T + chunk_size - 1) // chunk_size
    padded_T = num_chunks * chunk_size
    pad = padded_T - T

    if pad > 0:
        q = F.pad(q, (0, 0, 0, 0, 0, pad))
        k = F.pad(k, (0, 0, 0, 0, 0, pad))
        v = F.pad(v, (0, 0, 0, 0, 0, pad))
        g = F.pad(g, (0, 0, 0, 0, 0, pad), value=0.0)
        beta = F.pad(beta, (0, 0, 0, pad), value=0.0)

    C = chunk_size
    N = num_chunks
    device = q.device

    q = q.reshape(B, N, C, H, K).permute(0, 3, 1, 2, 4).contiguous()
    k = k.reshape(B, N, C, H, K).permute(0, 3, 1, 2, 4).contiguous()
    v = v.reshape(B, N, C, H, V).permute(0, 3, 1, 2, 4).contiguous()
    g = g.reshape(B, N, C, H, K).permute(0, 3, 1, 2, 4).contiguous()
    beta = beta.reshape(B, N, C, H).permute(0, 3, 1, 2).contiguous()

    G = g.cumsum(dim=-2)  # [B, H, N, C, K]

    # Relative decay difference: G_i - G_j
    G_diff = G.unsqueeze(-2) - G.unsqueeze(-3)  # [B, H, N, C, C, K]

    # Broadcastable causal masks
    mask_strict_lower = torch.tril(
        torch.ones(C, C, device=device, dtype=torch.bool), diagonal=-1
    ).view(1, 1, 1, C, C, 1)

    mask_lower = torch.tril(
        torch.ones(C, C, device=device, dtype=torch.bool), diagonal=0
    ).view(1, 1, 1, C, C, 1)

    # 1. Mask G_diff BEFORE exp to prevent exp(large_positive) -> Inf -> NaN in autograd
    G_diff_M = torch.where(mask_strict_lower, G_diff, -1e9)
    decay_M = torch.exp(G_diff_M)

    # 2. Key-Key interaction matrix M_ij
    k_i = k.unsqueeze(-2)
    k_j = k.unsqueeze(-3)
    M_raw = torch.sum(k_i * decay_M * k_j, dim=-1)

    beta_j = beta.unsqueeze(-2)
    M = M_raw * beta_j

    # 3. Solve A = (I + M)^(-1)
    I_C = torch.eye(C, device=device, dtype=torch.float32)
    I_plus_M = I_C + M
    A = torch.linalg.solve_triangular(
        I_plus_M, I_C.expand_as(I_plus_M), upper=False
    )

    # 4. Compute chunk write and read intermediates
    gated_k = torch.exp(G) * k
    W = torch.einsum("bhnij,bhnjk->bhnik", A, gated_k)
    U = torch.einsum("bhnij,bhnjv->bhniv", A, v)

    # 5. Query-Key causal interaction matrix Aqk
    G_diff_Aqk = torch.where(mask_lower, G_diff, -1e9)
    decay_Aqk = torch.exp(G_diff_Aqk)

    q_i = q.unsqueeze(-2)
    Aqk_raw = torch.sum(q_i * decay_Aqk * k_j, dim=-1)
    Aqk = Aqk_raw * beta_j

    # 6. Recurrent state propagation across chunks
    if initial_state is None:
        state = torch.zeros(B, H, K, V, device=device, dtype=torch.float32)
    else:
        state = initial_state.float()

    outputs = []

    for n in range(N):
        q_n = q[:, :, n]
        k_n = k[:, :, n]
        g_n = G[:, :, n]
        beta_n = beta[:, :, n]
        W_n = W[:, :, n]
        U_n = U[:, :, n]
        Aqk_n = Aqk[:, :, n]

        E_n = U_n - torch.einsum("bhck,bhkv->bhcv", W_n, state)

        gated_q = torch.exp(g_n) * q_n
        out_n = torch.einsum("bhck,bhkv->bhcv", gated_q, state) + torch.einsum(
            "bhij,bhjv->bhiv", Aqk_n, E_n
        )
        outputs.append(out_n)

        G_n_last = g_n[:, :, -1, :]
        state = state * torch.exp(G_n_last).unsqueeze(-1)

        decay_to_end = torch.exp(G_n_last.unsqueeze(-2) - g_n)
        k_decayed_beta = decay_to_end * k_n * beta_n.unsqueeze(-1)
        state = state + torch.einsum("bhck,bhcv->bhkv", k_decayed_beta, E_n)

    out = torch.cat(outputs, dim=2)
    out = out.transpose(1, 2).contiguous()[:, :T]

    return out.to(orig_dtype), state.to(orig_dtype)

## Dense FeedForward layer
class SwiGLUFFN(nn.Module):
    """SwiGLU Feed-Forward Network."""
    def __init__(self, d_model: int, ffn_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, ffn_dim, bias=False)  # Gate
        self.w2 = nn.Linear(ffn_dim, d_model, bias=False)  # Down
        self.w3 = nn.Linear(d_model, ffn_dim, bias=False)  # Up

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## Kimi's KDA
class KimiDeltaAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int = 16,
        head_dim_k: int = 64,
        head_dim_v: int = 64,
        low_rank_dim: int = 64,
        g_min: float = -5.0,
        chunk_size: int = 64
    ):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim_k = head_dim_k
        self.head_dim_v = head_dim_v
        self.g_min = g_min
        self.chunk_size = chunk_size

        inner_dim_k = num_heads * head_dim_k
        inner_dim_v = num_heads * head_dim_v

        # 1. Projections & ShortConvs
        self.q_proj = nn.Linear(d_model, inner_dim_k, bias=False)
        self.k_proj = nn.Linear(d_model, inner_dim_k, bias=False)
        self.v_proj = nn.Linear(d_model, inner_dim_v, bias=False)

        self.conv_q = ShortConv1d(inner_dim_k)
        self.conv_k = ShortConv1d(inner_dim_k)
        self.conv_v = ShortConv1d(inner_dim_v)

        # 2. Write Strength Beta Projection
        self.beta_proj = nn.Linear(d_model, num_heads, bias=False)

        # 3. Fine-grained Decay Logits (Low-Rank Parameterization)
        self.alpha_down = nn.Linear(d_model, low_rank_dim, bias=False)
        self.alpha_up = nn.Linear(low_rank_dim, inner_dim_k, bias=False)
        self.b_alpha = nn.Parameter(torch.zeros(num_heads, head_dim_k))

        # Learnable log-scale A_h initialized to 0
        self.A_h = nn.Parameter(torch.zeros(num_heads, head_dim_k))

        # 4. Low-Rank Output Gate & Head Normalization
        self.g_proj_down = nn.Linear(d_model, low_rank_dim, bias=False)
        self.g_proj_up = nn.Linear(low_rank_dim, inner_dim_v, bias=False)

        self.head_norm = RMSNorm(head_dim_v)
        self.out_proj = nn.Linear(inner_dim_v, d_model, bias=False)

    def forward(self, x: torch.Tensor, layer_state: dict = None):
        B, T, _ = x.shape

        # Extract states if provided
        cs_q = layer_state["conv_q"] if layer_state is not None else None
        cs_k = layer_state["conv_k"] if layer_state is not None else None
        cs_v = layer_state["conv_v"] if layer_state is not None else None
        rnn_state = layer_state["rnn"] if layer_state is not None else None

        # --- Compute Queries, Keys, and Values through Convs ---
        q_conv, next_cs_q = self.conv_q(self.q_proj(x), cs_q)
        k_conv, next_cs_k = self.conv_k(self.k_proj(x), cs_k)
        v_conv, next_cs_v = self.conv_v(self.v_proj(x), cs_v)

        q = F.silu(q_conv)
        k = F.silu(k_conv)
        v = F.silu(v_conv)

        # Reshape to (B, T, H, d_k) and (B, T, H, d_v)
        q = q.view(B, T, self.num_heads, self.head_dim_k)
        k = k.view(B, T, self.num_heads, self.head_dim_k)
        v = v.view(B, T, self.num_heads, self.head_dim_v)

        # Apply L2 norm per-head
        q = l2_norm(q)
        k = l2_norm(k)

        # Write strength beta_t in (0, 1)
        beta = torch.sigmoid(self.beta_proj(x)).unsqueeze(-1)  # (B, T, H, 1)

        # Fine-grained decay logits z_t
        z = self.alpha_up(self.alpha_down(x)).view(B, T, self.num_heads, self.head_dim_k) + self.b_alpha

        # --- Lower-Bounded Log-Decay (Kimi K3 Formula) ---
        scale = torch.exp(self.A_h).unsqueeze(0).unsqueeze(0)  # (1, 1, H, d_k)
        g_t = self.g_min * torch.sigmoid(scale * z)

        g_t = torch.clamp(g_t, min=self.g_min, max=-1e-4)

        if T > self.chunk_size:
            o_stacked, next_rnn_state = kda_chunkwise(
                q=q,
                k=k,
                v=v,
                g=g_t,
                beta=beta,
                initial_state=rnn_state,
                chunk_size=self.chunk_size,
                q_scale=self.head_dim_k ** -0.5,
            )
        else:
            o_stacked, next_rnn_state = kda_recurrent(
                q=q,
                k=k,
                v=v,
                g=g_t,
                beta=beta,
                initial_state=rnn_state,
                q_scale=self.head_dim_k ** -0.5,
            )

        # --- Output Gating & RMSNorm ---
        o_norm = self.head_norm(o_stacked)
        o_flat = o_norm.view(B, T, self.num_heads * self.head_dim_v)

        # Apply low-rank projection: Sigmoid(W_g_up(W_g_down(x)))
        gate_logits = self.g_proj_up(self.g_proj_down(x)) 

        gate = torch.sigmoid(gate_logits) 
        gated_out = gate * o_flat

        # Package the unified layer state for the next step
        next_layer_state = {
            "conv_q": next_cs_q,
            "conv_k": next_cs_k,
            "conv_v": next_cs_v,
            "rnn": next_rnn_state
        }

        return self.out_proj(gated_out), next_layer_state

## Decoder block
class KDABlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, head_dim_k: int, head_dim_v: int, ffn_dim: int):
        super().__init__()

        self.attn = KimiDeltaAttention(
            d_model=d_model,
            num_heads=num_heads,
            head_dim_k=head_dim_k,
            head_dim_v=head_dim_v
        )
        self.attn_norm = RMSNorm(d_model)
        self.ffn_norm = RMSNorm(d_model)

        # FeedForward Init
        self.ffn = SwiGLUFFN(d_model=d_model, ffn_dim=ffn_dim)

    def forward(self, x: torch.Tensor, layer_state: dict = None):
        attn_out, next_layer_state = self.attn(self.attn_norm(x), layer_state=layer_state)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, next_layer_state

## Main Model
class MimiKoKo_D1(nn.Module):
    def __init__(self, args):
        super().__init__()
        vocab_size = args.vocab_size
        d_model = args.d_model
        self.num_layers = args.num_layers

        self.embed_tokens = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            KDABlock(
                d_model=d_model,
                num_heads=args.num_heads,
                head_dim_k=args.head_dim,
                head_dim_v=args.head_dim,
                ffn_dim=args.ff_hidden_dim
            )
            for _ in range(self.num_layers)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)
        self.lm_head.weight = self.embed_tokens.weight

        self._apply_scaled_residual_init()

    def _init_weights(self, module):
        std = 0.02
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if getattr(module, "padding_idx", None) is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.Conv1d):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias) 

    def _apply_scaled_residual_init(self):
        std = 0.02
        scaled_std = std / math.sqrt(2.0 * self.num_layers)
        for name, param in self.named_parameters():
            if name.endswith("out_proj.weight") or name.endswith("w2.weight"):
                nn.init.normal_(param, mean=0.0, std=scaled_std) 
            if name.endswith("g_proj_up.weight"):
                nn.init.normal_(param, mean=0.0, std=0.001)

    def forward(self, input_ids: torch.Tensor, states: list = None):
        x = self.embed_tokens(input_ids)
        new_states = []

        for i, layer in enumerate(self.layers):
            layer_state = states[i] if states is not None else None
            x, next_state = layer(x, layer_state=layer_state)
            new_states.append(next_state)

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits, new_states



In [ ]:
## Train

import time
import yaml
from torch.utils.data import DataLoader
from dataclasses import asdict
import torch.nn.functional as F
from torch.optim import AdamW


torch.cuda.empty_cache()
torch.manual_seed(123)

paths = ProjectPaths()
model_args = ModelArgs()
train_args = TrainingArgs()


print(f"Training Device: {model_args.device}")

os.makedirs(paths.model_checkpoints_dir, exist_ok=True)

# Instantiate datasets and generators
dataset = TextTokenDataset(paths.data_corpus, model_args.context_length, paths.tokenizer_dir)
dataloader = DataLoader(dataset, batch_size=train_args.batch_size, shuffle=True, drop_last=True)

# ## Setting MoE args
# model_args.moe_args = MoEArgs(
#     num_experts=4,
#     top_k_experts=1,
#     hidden_dim_multiplier_perexp=1,
#     num_shared_experts=1,
#     use_shared_expert=True
# )

model_args.vocab_size = dataset.safe_vocab_size
print(model_args)

model = MimiKoKo_D1(model_args).to(model_args.device)

## Param count
tot_param = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Total parameters: {tot_param:.2f}M")
# if model_args.moe_args:
#     moe_param_count = calculate_active_params_MoE(model)
#     active_param = moe_param_count['active_parameters']
#     shared_param = moe_param_count['shared_expert_parameters'] if "shared_expert_parameters" in moe_param_count else 0
#     print(f"Active: {active_param}M | Shared: {shared_param}M")
#     # print(moe_param_count)

# Optimizer with Cosine scheduler
optimizer = AdamW(model.parameters(), lr=train_args.lr_rate, betas=(0.9, 0.990))

updates_per_epoch = math.ceil(
    len(dataloader) / train_args.grad_accum_steps
)
total_updates = (updates_per_epoch) * train_args.epochs
total_steps = total_updates//2

scheduler = get_cosine_schedule_with_warmup(optimizer, train_args.warmup_steps, total_updates)

steps_per_epoch = len(dataloader)
print(f"{steps_per_epoch} | {total_updates}")

model.train()
st_time = time.perf_counter()
net_step_time = 0
step_time = 0
global_step = 0

print("Beginning Training Iterations...\n")

for ep in range(train_args.epochs):
    for batch_idx, (x, y) in enumerate(dataloader):
        t0 = time.perf_counter()
        x, y = x.to(model_args.device), y.to(model_args.device)

        # Autocast automatically manages safe dtype casting
        with torch.autocast(device_type=model_args.device.type, dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, enabled=model_args.device.type == "cuda"):
            logits, __ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            # if model_args.moe_args:
            #     loss = loss + model.get_aux_loss()

        loss = loss / train_args.grad_accum_steps
        # scaler.scale(loss).backward()
        loss.backward()

        if (batch_idx + 1) % train_args.grad_accum_steps == 0:
            nn.utils.clip_grad_norm_(model.parameters(), train_args.grad_clip)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        global_step += 1

        step_time = time.perf_counter() - t0
        net_step_time += step_time

        if batch_idx % 10 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch: {ep} (BID:{batch_idx}) | Step: {global_step}/{total_steps} | Step_time: {step_time:.6f} s | Loss: {loss.item() * train_args.grad_accum_steps:.4f} | LR: {current_lr:.6f}")

        # Save Checkpoints
        if global_step > 0 and global_step % train_args.save_every_steps == 0:
            checkpoint_path = os.path.join(paths.model_checkpoints_dir, f"mla_step_{global_step}.pt")
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint model artifact at: {checkpoint_path}")

        if global_step >= total_steps:
            break

        if train_args.tot_steps > 1:
            if batch_idx >= train_args.tot_steps:
                print(f"{train_args.tot_steps} steps reached")
                break

    preds = torch.argmax(logits, dim=-1)
    correct_mask = (preds == y)
    total_correct = correct_mask.sum().item()
    total_tokens = y.numel()
    print(f"Epoch:{ep} | Accuracy: {(total_correct/total_tokens) * 100}")

tot_time = time.perf_counter() - st_time
last_loss = loss.item()
print(f"\nTraining completed successfully in: {tot_time:.5f}s")

# Final Model Storage Integration
model_final_path = os.path.join(paths.model_checkpoints_dir, "MimiKoKo_D1_v1.pt")
config_path = os.path.join(paths.model_checkpoints_dir, 'training_config-MimiKoKo_D1_v1.yaml')

model_args.device = str(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

training_params = {
    "hyperparameters": asdict(model_args),
    "training_args": asdict(train_args),
    "training_stats": {
        'total_steps': total_steps,
        'step_time_s': float(f"{step_time:.6f}"),
        'loss': float(f"{last_loss:.6f}"),
        'training_time_s': float(f"{tot_time:.4f}")
    },
    "total_param": f"{tot_param:.2f}M",
    # "active_param": f"{active_param:.2f}M" if model_args.moe_args else None
}

try:
    with open(config_path, 'w') as file:
        yaml.dump(training_params, file)
except Exception as exc:
    print(f"Error saving config file! \n{exc}")

torch.save(model.state_dict(), model_final_path)
print(f"Model artifacts saved at: {model_final_path}")

del model

"""
At Total parameters: 62.87M, and (vocab_size=3216, d_model=1024, num_layers=4, num_heads=8, context_length=1024, head_dim=64, ff_hidden_dim=4096,)

GPU: 7.1 GB
System RAM: 5.3 GB

"""


Training Device: cuda
ModelArgs(vocab_size=3216, d_model=1024, num_layers=4, num_heads=8, context_length=1024, head_dim=64, ff_hidden_dim=4096, apply_rope=False, use_checkpointing=False, device=device(type='cuda'))
Total parameters: 64.58M
8960 | 8960
Beginning Training Iterations...

Epoch: 0 (BID:0) | Step: 1/4480 | Step_time: 1.256812 s | Loss: 8.1495 | LR: 0.000005
Epoch: 0 (BID:10) | Step: 11/4480 | Step_time: 0.644438 s | Loss: 7.3961 | LR: 0.000055
Epoch: 0 (BID:20) | Step: 21/4480 | Step_time: 0.643852 s | Loss: 5.0798 | LR: 0.000105
Epoch: 0 (BID:30) | Step: 31/4480 | Step_time: 0.658021 s | Loss: 5.1630 | LR: 0.000155
Epoch: 0 (BID:40) | Step: 41/4480 | Step_time: 0.659267 s | Loss: 3.9718 | LR: 0.000205
Epoch: 0 (BID:50) | Step: 51/4480 | Step_time: 0.667178 s | Loss: 5.0581 | LR: 0.000255
Epoch: 0 (BID:60) | Step: 61/4480 | Step_time: 0.678887 s | Loss: 1.8603 | LR: 0.000305
Epoch: 0 (BID:70) | Step: 71/4480 | Step_time: 0.683660 s | Loss: 3.1040 | LR: 0.000355
Epoch: 0 (BI

### Inference  


In [ ]:
## Init

import os
import time
import yaml
import torch
from transformers import AutoTokenizer
import torch.nn as nn
import torch.nn.functional as F

torch.cuda.empty_cache()

class KDAStatefulInferenceEngine:
    def __init__(self, model: torch.nn.Module, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device
        self.model.eval()

    @torch.no_grad()
    def sample(self, logits: torch.Tensor, temperature: float = 1.0, min_p: float = 0.0) -> torch.Tensor:
        if temperature <= 0.0:
            # Greedy decoding
            return torch.argmax(logits, dim=-1, keepdim=True)

        logits = logits / max(temperature, 1e-3)
        probs = F.softmax(logits, dim=-1)

        if min_p > 0.0:
            p_max = probs.max(dim=-1, keepdim=True).values
            indices_to_remove = probs < (min_p * p_max)
            logits[indices_to_remove] = float('-inf')
            probs = F.softmax(logits, dim=-1)

        _max, _ = probs.max(dim=-1, keepdim=True)
        token_conf = _max.item()
        if token_conf <= 0.95:
            print(f"<<< low  conf: {token_conf:.6f} >>>")

        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate(self, prompt: str, max_new_tokens: int = 256, temperature: float = 0.5, min_p: float = 0.1):
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.device)

        # --- Prefill Phase ---
        # Hack for Linear Attention: For long prompts, chunkwise evaluation is strictly faster.
        # Since your chunkwise is tied to `model.training = True`, we temporarily toggle it
        # for prefill to benefit from parallel chunk processing.
        
        # seq_len = input_ids.shape[1]
        # if seq_len > 64:
        #     self.model.train()

        self.model.eval() # Ensure we are back in evaluation/recurrent mode 

        logits, states = self.model(input_ids, states=None) 

        # Get the first token
        next_token_logits = logits[:, -1, :]
        next_token = self.sample(next_token_logits, temperature, min_p)
        generated_tokens = [next_token.item()]

        # --- Decoding Phase (O(1) complexity per step) ---
        for _ in range(max_new_tokens - 1):
            # Pass ONLY the single newest token [1, 1] and the accumulated state dictionaries
            logits, states = self.model(next_token, states=states)

            # Extract logit for the single step evaluated
            next_token_logits = logits[:, -1, :]
            next_token = self.sample(next_token_logits, temperature, min_p)
            token_id = next_token.item()

            generated_tokens.append(token_id)

            # Stop condition
            if self.tokenizer.eos_token_id is not None and token_id == self.tokenizer.eos_token_id:
                break

        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True), len(generated_tokens)


paths = ProjectPaths()

# 2. Load tokenizer
if not os.path.exists(paths.tokenizer_dir):
    raise FileNotFoundError(f"Tokenizer not found at {paths.tokenizer_dir}")

tokenizer = AutoTokenizer.from_pretrained(paths.tokenizer_dir)

# 4. Load weights and configs
ckpt_path = os.path.join(paths.model_checkpoints_dir, "MimiKoKo_D1_v1.pt")
config_path = os.path.join(paths.model_checkpoints_dir, "training_config-MimiKoKo_D1_v1.yaml")

if not os.path.exists(config_path):
    raise FileNotFoundError(f"Configuration file not found at {config_path}. Did you run train.py first?")

with open(config_path, 'r') as file:
    configs = yaml.safe_load(file)

hyperparams = configs["hyperparameters"]
model_args = ModelArgs(**hyperparams)
device = model_args.device
print(f"Model args >>> \n{model_args.__dict__}")
print(f"Device: {device}")

# 3. Initialize Model using Config
model = MimiKoKo_D1(model_args).to(model_args.device)

if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    # Handle cases where state_dict is nested or flat
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict, strict=False)
    print(f"Successfully loaded weights from {ckpt_path}")
else:
    print(f"Warning: Checkpoint not found at {ckpt_path}. Running with random initialization.")

# 5. Initialize Stateful Generator Engine
engine = KDAStatefulInferenceEngine(model, tokenizer)


In [ ]:
# 6. Generation
prompts = [
    "We are accounted",
    # "A freight train",
    "Researchers monitoring tidal",
    "A regional health network"
]

# prompts = [
#     "SAMPLE 3: JavaScript Debounce",
#     "SAMPLE 6: Docker Multi-Stage Build",
#     # "Explain binary search with Python ",
#     "### SAMPLE 9: Rust Ownership"
# ]

# prompts = [
#     "Question: A factory produced 130 items. 38% passed inspection ",
#     "Our study focused on static, 2D ",
#     """ "Text": "7. Acknowledgment\n We gratefully """
# ]

for prompt in prompts:
    print(f"\n[Prompt]: {prompt}")

    gen_st_time = time.perf_counter()
    generated_text, token_len = engine.generate(
        prompt.strip(),
        max_new_tokens=256,
        temperature=0.5,
        min_p=0.1
    )

    tot_gen_time = time.perf_counter() - gen_st_time

    # 7. Results
    print(f"[Generated]: {generated_text}\n")
    print(f"Inference time: {tot_gen_time:.5f}s")
    if tot_gen_time > 0.0:
        print(f"{int(token_len/tot_gen_time)} tokens/sec \n")

